# OrcaFlex batch simulations — JONSWAP, damping cases en post-processing

Notebook voor het automatisch runnen van OrcaFlex simulaties met meerdere `Hs/Tp` combinaties, 4 wave directions, 4 damping cases, output voor `Heave`, `Roll`, `Pitch`, berekening van `STD`, `MPM`, plots van de eerste 100 seconden en een duidelijke outputstructuur.

> Pas eerst de paden en OrcaFlex veldnamen aan in de configuratiecel. Draai daarna de cellen van boven naar beneden.

## 1. Imports

In [35]:
from pathlib import Path
import math
import traceback

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import OrcFxAPI

## 2. Projectconfiguratie

Pas vooral `PROJECT_DIR` en `BASE_MODEL_PATH` aan naar jouw lokale mapstructuur.

In [36]:
# =========================
# Project paths
# =========================


PROJECT_DIR = Path(r"C:\Users\verav\Desktop\Studie\Afstuderen\PHASE2_ORCA")
BASE_MODEL_PATH = PROJECT_DIR / "Merganserfinal_constrained.dat"

INPUT_DIR = PROJECT_DIR / "input"
RESULTS_DIR = PROJECT_DIR / "results"
PLOTS_DIR = PROJECT_DIR / "plots"
TEMP_DIR = PROJECT_DIR / "temp"

SEA_STATES_FILE = INPUT_DIR / "sea_states.xlsx"
DAMPING_CASES_FILE = INPUT_DIR / "damping_cases.xlsx"

# =========================
# OrcaFlex object names
# =========================

VESSEL_NAME = "floaters"
VESSEL_TYPE_NAME = "floatertype"

# =========================
# Simulation setup
# =========================

WAVE_DIRECTIONS = [45]

SIMULATION_DURATION = 10800.0  # 3 hours [s]
ANALYSIS_START = 600.0         # exclude first 600 s
ANALYSIS_END = 10800.0

ANALYSIS_START = 100.0         # exclude first 600 s
ANALYSIS_END = 300


PLOT_START = 100.0
PLOT_END = 300.0
TIME_TRACE_PLOT_DIRECTION = 45  # only make first-100s comparison plots for this direction

SAVE_TEMP_DAT_FILES = False
SAVE_SIM_FILES = False

# Assumption:
# Z          = Heave
# Rotation 1 = Roll
# Rotation 2 = Pitch
DOF_RESULT_MAP = {
    "Heave": "Z",
    "Roll": "Rotation 1",
    "Pitch": "Rotation 2",
}

DOF_UNITS = {
    "Heave": "m",
    "Roll": "deg",
    "Pitch": "deg",
}

## 3. Mappen aanmaken

In [37]:
def ensure_directories():
    INPUT_DIR.mkdir(parents=True, exist_ok=True)
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    PLOTS_DIR.mkdir(parents=True, exist_ok=True)
    TEMP_DIR.mkdir(parents=True, exist_ok=True)

    (PROJECT_DIR / "model").mkdir(parents=True, exist_ok=True)
    (PLOTS_DIR / "time_traces_first_100s").mkdir(parents=True, exist_ok=True)
    (PLOTS_DIR / "std_comparison").mkdir(parents=True, exist_ok=True)
    (PLOTS_DIR / "mpm_comparison").mkdir(parents=True, exist_ok=True)
    (PLOTS_DIR / "damping_reduction").mkdir(parents=True, exist_ok=True)

ensure_directories()
print(f"Project folders checked/created under: {PROJECT_DIR}")

Project folders checked/created under: C:\Users\verav\Desktop\Studie\Afstuderen\PHASE2_ORCA


## 4. Optioneel: input-Excel templates aanmaken

Draai deze cel alleen als je de Excelbestanden nog niet hebt. Als de bestanden al bestaan, worden ze niet overschreven.

In [38]:
def create_input_templates(overwrite=False):
    sea_states_template = pd.DataFrame([
        {"sea_state_id": 1, "Hs": 2.0, "Tp": 8.0, "gamma": 3.3, "seed": 1},
       
    ])

    damping_cases_template = pd.DataFrame([
        {"damping_case": 1, "heave_lin": 0, "heave_quad": 0, "pitch_lin": 0, "pitch_quad": 0, "roll_lin": 0, "roll_quad": 0},
        {"damping_case": 2, "heave_lin": 7, "heave_quad": 0, "pitch_lin": 300, "pitch_quad": 0, "roll_lin": 300, "roll_quad": 0},
        {"damping_case": 3, "heave_lin": 0, "heave_quad": 5, "pitch_lin": 175, "pitch_quad": 1200, "roll_lin": 150, "roll_quad": 1400},
        {"damping_case": 4, "heave_lin": 4, "heave_quad": 4, "pitch_lin": 250, "pitch_quad": 1000, "roll_lin": 250, "roll_quad": 1000},
    ])

    if overwrite or not SEA_STATES_FILE.exists():
        with pd.ExcelWriter(SEA_STATES_FILE, engine="openpyxl") as writer:
            sea_states_template.to_excel(writer, sheet_name="sea_states", index=False)
        print(f"Created: {SEA_STATES_FILE}")
    else:
        print(f"Already exists, not overwritten: {SEA_STATES_FILE}")

    if overwrite or not DAMPING_CASES_FILE.exists():
        with pd.ExcelWriter(DAMPING_CASES_FILE, engine="openpyxl") as writer:
            damping_cases_template.to_excel(writer, sheet_name="damping_cases", index=False)
        print(f"Created: {DAMPING_CASES_FILE}")
    else:
        print(f"Already exists, not overwritten: {DAMPING_CASES_FILE}")

create_input_templates(overwrite=False)

Already exists, not overwritten: C:\Users\verav\Desktop\Studie\Afstuderen\PHASE2_ORCA\input\sea_states.xlsx
Already exists, not overwritten: C:\Users\verav\Desktop\Studie\Afstuderen\PHASE2_ORCA\input\damping_cases.xlsx


## 5. Inputtabellen inlezen en controleren

In [39]:
def read_inputs():
    sea_states = pd.read_excel(SEA_STATES_FILE, sheet_name="sea_states")
    damping_cases = pd.read_excel(DAMPING_CASES_FILE, sheet_name="damping_cases")

    required_sea_cols = ["sea_state_id", "Hs", "Tp", "gamma", "seed"]
    required_damping_cols = [
        "damping_case",
        "heave_lin", "heave_quad",
        "pitch_lin", "pitch_quad",
        "roll_lin", "roll_quad",
    ]

    missing_sea = [c for c in required_sea_cols if c not in sea_states.columns]
    missing_damping = [c for c in required_damping_cols if c not in damping_cases.columns]

    if missing_sea:
        raise ValueError(f"Missing columns in sea_states.xlsx: {missing_sea}")
    if missing_damping:
        raise ValueError(f"Missing columns in damping_cases.xlsx: {missing_damping}")

    return sea_states, damping_cases

sea_states, damping_cases = read_inputs()
print("Sea states:")
display(sea_states)
print("Damping cases:")
display(damping_cases)

Sea states:


,sea_state_id,Hs,Tp,gamma,seed
0,1,2,8,3.3,1


Damping cases:


,damping_case,heave_lin,heave_quad,pitch_lin,pitch_quad,roll_lin,roll_quad
0,1,0,0,0,0,0,0
1,2,7,0,300,0,300,0
2,3,0,5,175,1200,150,1400
3,4,4,4,250,1000,250,1000


## 6. Algemene helperfuncties

In [40]:
def safe_name(value):
    return str(value).replace(".", "p").replace("-", "m")


def case_label(sea_state_id, hs, tp, direction, damping_case):
    return (
        f"SS{int(sea_state_id):03d}_"
        f"Hs_{safe_name(hs)}_"
        f"Tp_{safe_name(tp)}_"
        f"Dir_{int(direction):03d}_"
        f"Damp_{int(damping_case):02d}"
    )


def log_message(message):
    log_file = RESULTS_DIR / "run_log.txt"
    with open(log_file, "a", encoding="utf-8") as f:
        f.write(str(message) + "")
    print(message)

## 7. OrcaFlex model configureren

Deze functies zetten de simulatie-instellingen, JONSWAP golfconditie en damping case.

In [41]:
def configure_general_simulation_settings(model):
    model.general.StageDuration[0] = SIMULATION_DURATION


def configure_jonswap_wave(model, hs, tp, gamma, direction, seed):
    # Assumes one wave train: model.environment.WaveTrains[0]
    wave_train = model.environment

    # wave_train.WaveType = "JONSWAP"
    wave_train.WaveHs = float(hs)
    wave_train.WaveTz = float(tp)
    wave_train.WaveGamma = float(gamma)
    wave_train.WaveDirection = float(direction)
    # wave_train.WaveSeed = int(seed)


def apply_damping_case(model, damping_row):
    vesseltype = model[VESSEL_TYPE_NAME]

    # Heave
    vesseltype.OtherDampingLinearCoeffz = float(damping_row["heave_lin"])
    vesseltype.OtherDampingQuadraticCoeffz = float(damping_row["heave_quad"])

    # Pitch
    vesseltype.OtherDampingLinearCoeffRy = float(damping_row["pitch_lin"])
    vesseltype.OtherDampingQuadraticCoeffRy = float(damping_row["pitch_quad"])

    # Roll
    vesseltype.OtherDampingLinearCoeffRx = float(damping_row["roll_lin"])
    vesseltype.OtherDampingQuadraticCoeffRx = float(damping_row["roll_quad"])

## 8. Simulatie runnen en resultaten uitlezen

In [42]:
def run_model(model):
    # model.CalculateStatics()
    model.RunSimulation()


def get_time_history(model, dof_name, period_start, period_end):
    vessel = model[VESSEL_NAME]
    result_name = DOF_RESULT_MAP[dof_name]

    period = OrcFxAPI.SpecifiedPeriod(period_start, period_end)
    time = np.array(model.SampleTimes(period))
    values = np.array(vessel.TimeHistory(result_name, period))

    return time, values

## 9. STD en MPM berekenen

Voor de MPM gebruiken we als eerste engineering-aanpak:

$$MPM = \mu \pm \sigma \sqrt{2 \ln(N)}$$

met:

$$N = \frac{T_\text{analysis}}{T_p}$$

Later kun je dit verfijnen door response peaks uit de tijdreeks te detecteren.

In [43]:
def calculate_statistics(values, tp, analysis_duration):
    values = np.asarray(values)

    mean = float(np.mean(values))
    std = float(np.std(values, ddof=0))

    n_cycles = max(analysis_duration / float(tp), 1.0)
    factor = math.sqrt(2.0 * math.log(n_cycles))

    mpm_positive = mean + std * factor
    mpm_negative = mean - std * factor
    mpm_abs = max(abs(mpm_positive), abs(mpm_negative))

    max_abs_simulated = float(np.max(np.abs(values)))

    return {
        "mean": mean,
        "std": std,
        "n_cycles": n_cycles,
        "mpm_positive": float(mpm_positive),
        "mpm_negative": float(mpm_negative),
        "mpm_abs": float(mpm_abs),
        "max_abs_simulated": max_abs_simulated,
    }

## 10. Eén case runnen

Deze functie wordt straks door de batch-loop aangeroepen.

In [44]:
def run_single_case(sea_row, damping_row, direction):
    sea_state_id = int(sea_row["sea_state_id"])
    hs = float(sea_row["Hs"])
    tp = float(sea_row["Tp"])
    gamma = float(sea_row["gamma"])
    seed = int(sea_row["seed"])
    damping_case = int(damping_row["damping_case"])

    label = case_label(sea_state_id, hs, tp, direction, damping_case)
    log_message(f"Running {label}")

    model = OrcFxAPI.Model(str(BASE_MODEL_PATH))

    # configure_general_simulation_settings(model)
    configure_jonswap_wave(model, hs, tp, gamma, direction, seed)
    apply_damping_case(model, damping_row)

    temp_dat_path = TEMP_DIR / f"{label}.dat"
    temp_sim_path = TEMP_DIR / f"{label}.sim"

    if SAVE_TEMP_DAT_FILES:
        model.SaveData(str(temp_dat_path))

    run_model(model)

    if SAVE_SIM_FILES:
        model.SaveSimulation(str(temp_sim_path))

    analysis_duration = ANALYSIS_END - ANALYSIS_START
    rows = []
    plot_traces = []

    for dof in DOF_RESULT_MAP:
        _, values_analysis = get_time_history(model, dof, ANALYSIS_START, ANALYSIS_END)
        stats = calculate_statistics(values_analysis, tp, analysis_duration)

        rows.append({
            "case_label": label,
            "sea_state_id": sea_state_id,
            "Hs": hs,
            "Tp": tp,
            "gamma": gamma,
            "seed": seed,
            "direction": direction,
            "damping_case": damping_case,
            "dof": dof,
            "unit": DOF_UNITS.get(dof, ""),
            "analysis_start_s": ANALYSIS_START,
            "analysis_end_s": ANALYSIS_END,
            **stats,
            "status": "OK",
            "error_message": "",
        })

        if direction == TIME_TRACE_PLOT_DIRECTION:
            time_plot, values_plot = get_time_history(model, dof, PLOT_START, PLOT_END)
            plot_traces.append({
                "sea_state_id": sea_state_id,
                "Hs": hs,
                "Tp": tp,
                "direction": direction,
                "damping_case": damping_case,
                "dof": dof,
                "time": time_plot,
                "values": values_plot,
            })

    return rows, plot_traces

## 11. Batch-run uitvoeren

Deze cel draait alle combinaties van sea states, wave directions en damping cases. Bij een error wordt de fout gelogd en gaat de batch door met de volgende case.

In [45]:
all_result_rows = []
time_trace_store = []
error_rows = []

for _, sea_row in sea_states.iterrows():
    for direction in WAVE_DIRECTIONS:
        for _, damping_row in damping_cases.iterrows():
            try:
                result_rows, plot_traces = run_single_case(sea_row, damping_row, direction)
                all_result_rows.extend(result_rows)
                time_trace_store.extend(plot_traces)

            except Exception as exc:
                sea_state_id = int(sea_row["sea_state_id"])
                hs = float(sea_row["Hs"])
                tp = float(sea_row["Tp"])
                damping_case = int(damping_row["damping_case"])
                label = case_label(sea_state_id, hs, tp, direction, damping_case)

                error_text = traceback.format_exc()
                log_message(f"ERROR in {label}: {exc}")
                log_message(error_text)

                error_rows.append({
                    "case_label": label,
                    "sea_state_id": sea_state_id,
                    "Hs": hs,
                    "Tp": tp,
                    "direction": direction,
                    "damping_case": damping_case,
                    "status": "ERROR",
                    "error_message": str(exc),
                })

summary_df = pd.DataFrame(all_result_rows)
errors_df = pd.DataFrame(error_rows)

print(f"Finished. Successful result rows: {len(summary_df)}")
print(f"Errors: {len(errors_df)}")

display(summary_df.head())
if len(errors_df) > 0:
    display(errors_df)

Running SS001_Hs_2p0_Tp_8p0_Dir_045_Damp_01
Running SS001_Hs_2p0_Tp_8p0_Dir_045_Damp_02
Running SS001_Hs_2p0_Tp_8p0_Dir_045_Damp_03
Running SS001_Hs_2p0_Tp_8p0_Dir_045_Damp_04
Finished. Successful result rows: 12
Errors: 0


,case_label,sea_state_id,Hs,Tp,gamma,seed,direction,damping_case,dof,unit,...,analysis_end_s,mean,std,n_cycles,mpm_positive,mpm_negative,mpm_abs,max_abs_simulated,status,error_message
0,SS001_Hs_2p0_Tp_8p0_Dir_045_Damp_01,1,2.0,8.0,3.3,1,45,1,Heave,m,...,300,0.000119,1.126754,25.0,2.859001,-2.858763,2.859001,3.571595,OK,
1,SS001_Hs_2p0_Tp_8p0_Dir_045_Damp_01,1,2.0,8.0,3.3,1,45,1,Roll,deg,...,300,-0.078565,13.593214,25.0,34.411122,-34.568251,34.568251,33.186721,OK,
2,SS001_Hs_2p0_Tp_8p0_Dir_045_Damp_01,1,2.0,8.0,3.3,1,45,1,Pitch,deg,...,300,-0.169757,10.934614,25.0,27.574337,-27.913851,27.913851,32.491606,OK,
3,SS001_Hs_2p0_Tp_8p0_Dir_045_Damp_02,1,2.0,8.0,3.3,1,45,2,Heave,m,...,300,-0.000329,0.539632,25.0,1.368866,-1.369523,1.369523,1.487919,OK,
4,SS001_Hs_2p0_Tp_8p0_Dir_045_Damp_02,1,2.0,8.0,3.3,1,45,2,Roll,deg,...,300,0.131899,6.145077,25.0,15.723634,-15.459836,15.723634,16.003432,OK,


## 12. Resultaten opslaan naar Excel en CSV

In [46]:
summary_csv_path = RESULTS_DIR / "summary_all_cases.csv"
summary_xlsx_path = RESULTS_DIR / "summary_all_cases.xlsx"

summary_df.to_csv(summary_csv_path, index=False)

with pd.ExcelWriter(summary_xlsx_path, engine="openpyxl") as writer:
    summary_df.to_excel(writer, sheet_name="summary", index=False)
    if len(errors_df) > 0:
        errors_df.to_excel(writer, sheet_name="errors", index=False)

print(f"Saved: {summary_csv_path}")
print(f"Saved: {summary_xlsx_path}")

Saved: C:\Users\verav\Desktop\Studie\Afstuderen\PHASE2_ORCA\results\summary_all_cases.csv
Saved: C:\Users\verav\Desktop\Studie\Afstuderen\PHASE2_ORCA\results\summary_all_cases.xlsx


## 13. Time trace plots: eerste 100 seconden

Voor elke sea state, voor `TIME_TRACE_PLOT_DIRECTION`, wordt per DOF een plot gemaakt met alle damping cases over elkaar.

In [47]:
def plot_time_traces(time_trace_store):
    if not time_trace_store:
        print("No time traces stored.")
        return

    key_df = pd.DataFrame([
        {
            "sea_state_id": item["sea_state_id"],
            "Hs": item["Hs"],
            "Tp": item["Tp"],
            "direction": item["direction"],
            "dof": item["dof"],
        }
        for item in time_trace_store
    ]).drop_duplicates()

    for _, key in key_df.iterrows():
        subset = [
            item for item in time_trace_store
            if item["sea_state_id"] == key["sea_state_id"]
            and item["direction"] == key["direction"]
            and item["dof"] == key["dof"]
        ]

        plt.figure(figsize=(11, 6))
        for item in subset:
            plt.plot(item["time"], item["values"], label=f"Damping case {int(item['damping_case'])}", linewidth=1.2)

        dof = key["dof"]
        unit = DOF_UNITS.get(dof, "")
        plt.xlabel("Time [s]")
        plt.ylabel(f"{dof} [{unit}]")
        plt.title(f"{dof} first 100 s | Hs={key['Hs']}, Tp={key['Tp']}, Direction={int(key['direction'])}°")
        plt.grid(True)
        plt.legend()
        plt.tight_layout()

        filename = (
            f"SS{int(key['sea_state_id']):03d}_"
            f"Hs_{safe_name(key['Hs'])}_"
            f"Tp_{safe_name(key['Tp'])}_"
            f"Dir_{int(key['direction']):03d}_"
            f"{dof}_first_100s.png"
        )
        plt.savefig(PLOTS_DIR / "time_traces_first_100s" / filename, dpi=200)
        plt.close()

    print(f"Saved time trace plots in: {PLOTS_DIR / 'time_traces_first_100s'}")

plot_time_traces(time_trace_store)

Saved time trace plots in: C:\Users\verav\Desktop\Studie\Afstuderen\PHASE2_ORCA\plots\time_traces_first_100s


## 14. STD en MPM comparison plots

Deze plots vergelijken damping cases per sea state, direction en DOF.

In [48]:
def make_metric_comparison_plots(summary_df, metric, output_subfolder, ylabel):
    if summary_df.empty:
        print("summary_df is empty; no plots made.")
        return

    plot_dir = PLOTS_DIR / output_subfolder
    plot_dir.mkdir(parents=True, exist_ok=True)

    group_cols = ["sea_state_id", "Hs", "Tp", "direction", "dof"]
    keys = summary_df[group_cols].drop_duplicates()

    for _, key in keys.iterrows():
        subset = summary_df[
            (summary_df["sea_state_id"] == key["sea_state_id"]) &
            (summary_df["direction"] == key["direction"]) &
            (summary_df["dof"] == key["dof"])
        ].sort_values("damping_case")

        if subset.empty:
            continue

        plt.figure(figsize=(8, 5))
        plt.bar(subset["damping_case"].astype(str), subset[metric])
        plt.xlabel("Damping case")
        plt.ylabel(ylabel)
        plt.title(f"{metric} | {key['dof']} | Hs={key['Hs']}, Tp={key['Tp']}, Dir={int(key['direction'])}°")
        plt.grid(True, axis="y")
        plt.tight_layout()

        filename = (
            f"SS{int(key['sea_state_id']):03d}_"
            f"Hs_{safe_name(key['Hs'])}_"
            f"Tp_{safe_name(key['Tp'])}_"
            f"Dir_{int(key['direction']):03d}_"
            f"{key['dof']}_{metric}.png"
        )
        plt.savefig(plot_dir / filename, dpi=200)
        plt.close()

    print(f"Saved {metric} plots in: {plot_dir}")

make_metric_comparison_plots(summary_df, metric="std", output_subfolder="std_comparison", ylabel="STD")
make_metric_comparison_plots(summary_df, metric="mpm_abs", output_subfolder="mpm_comparison", ylabel="MPM abs")

Saved std plots in: C:\Users\verav\Desktop\Studie\Afstuderen\PHASE2_ORCA\plots\std_comparison
Saved mpm_abs plots in: C:\Users\verav\Desktop\Studie\Afstuderen\PHASE2_ORCA\plots\mpm_comparison


## 15. Reductie t.o.v. damping case 1

Hiermee zie je per damping case de verhouding t.o.v. de baseline zonder damping.

- `std_ratio_to_case_1 < 1` betekent reductie van STD
- `mpm_ratio_to_case_1 < 1` betekent reductie van MPM

In [49]:
def add_reduction_ratios(summary_df, baseline_case=1):
    if summary_df.empty:
        return summary_df

    index_cols = ["sea_state_id", "Hs", "Tp", "direction", "dof"]
    baseline = summary_df[summary_df["damping_case"] == baseline_case][index_cols + ["std", "mpm_abs"]].rename(
        columns={"std": "std_baseline", "mpm_abs": "mpm_abs_baseline"}
    )

    out = summary_df.merge(baseline, on=index_cols, how="left")
    out["std_ratio_to_case_1"] = out["std"] / out["std_baseline"]
    out["mpm_ratio_to_case_1"] = out["mpm_abs"] / out["mpm_abs_baseline"]
    return out

summary_with_ratios = add_reduction_ratios(summary_df, baseline_case=1)
display(summary_with_ratios.head())

ratio_csv_path = RESULTS_DIR / "summary_all_cases_with_ratios.csv"
ratio_xlsx_path = RESULTS_DIR / "summary_all_cases_with_ratios.xlsx"
summary_with_ratios.to_csv(ratio_csv_path, index=False)
with pd.ExcelWriter(ratio_xlsx_path, engine="openpyxl") as writer:
    summary_with_ratios.to_excel(writer, sheet_name="summary_with_ratios", index=False)

print(f"Saved: {ratio_csv_path}")
print(f"Saved: {ratio_xlsx_path}")

,case_label,sea_state_id,Hs,Tp,gamma,seed,direction,damping_case,dof,unit,...,mpm_positive,mpm_negative,mpm_abs,max_abs_simulated,status,error_message,std_baseline,mpm_abs_baseline,std_ratio_to_case_1,mpm_ratio_to_case_1
0,SS001_Hs_2p0_Tp_8p0_Dir_045_Damp_01,1,2.0,8.0,3.3,1,45,1,Heave,m,...,2.859001,-2.858763,2.859001,3.571595,OK,,1.126754,2.859001,1.000000,1.000000
1,SS001_Hs_2p0_Tp_8p0_Dir_045_Damp_01,1,2.0,8.0,3.3,1,45,1,Roll,deg,...,34.411122,-34.568251,34.568251,33.186721,OK,,13.593214,34.568251,1.000000,1.000000
2,SS001_Hs_2p0_Tp_8p0_Dir_045_Damp_01,1,2.0,8.0,3.3,1,45,1,Pitch,deg,...,27.574337,-27.913851,27.913851,32.491606,OK,,10.934614,27.913851,1.000000,1.000000
3,SS001_Hs_2p0_Tp_8p0_Dir_045_Damp_02,1,2.0,8.0,3.3,1,45,2,Heave,m,...,1.368866,-1.369523,1.369523,1.487919,OK,,1.126754,2.859001,0.478927,0.479022
4,SS001_Hs_2p0_Tp_8p0_Dir_045_Damp_02,1,2.0,8.0,3.3,1,45,2,Roll,deg,...,15.723634,-15.459836,15.723634,16.003432,OK,,13.593214,34.568251,0.452069,0.454858


Saved: C:\Users\verav\Desktop\Studie\Afstuderen\PHASE2_ORCA\results\summary_all_cases_with_ratios.csv
Saved: C:\Users\verav\Desktop\Studie\Afstuderen\PHASE2_ORCA\results\summary_all_cases_with_ratios.xlsx


In [50]:
def make_ratio_plots(summary_df, ratio_col, ylabel):
    if summary_df.empty:
        print("summary_df is empty; no ratio plots made.")
        return

    plot_dir = PLOTS_DIR / "damping_reduction"
    plot_dir.mkdir(parents=True, exist_ok=True)

    group_cols = ["sea_state_id", "Hs", "Tp", "direction", "dof"]
    keys = summary_df[group_cols].drop_duplicates()

    for _, key in keys.iterrows():
        subset = summary_df[
            (summary_df["sea_state_id"] == key["sea_state_id"]) &
            (summary_df["direction"] == key["direction"]) &
            (summary_df["dof"] == key["dof"])
        ].sort_values("damping_case")

        if subset.empty or ratio_col not in subset.columns:
            continue

        plt.figure(figsize=(8, 5))
        plt.bar(subset["damping_case"].astype(str), subset[ratio_col])
        plt.axhline(1.0, linestyle="--", linewidth=1)
        plt.xlabel("Damping case")
        plt.ylabel(ylabel)
        plt.title(f"{ratio_col} | {key['dof']} | Hs={key['Hs']}, Tp={key['Tp']}, Dir={int(key['direction'])}°")
        plt.grid(True, axis="y")
        plt.tight_layout()

        filename = (
            f"SS{int(key['sea_state_id']):03d}_"
            f"Hs_{safe_name(key['Hs'])}_"
            f"Tp_{safe_name(key['Tp'])}_"
            f"Dir_{int(key['direction']):03d}_"
            f"{key['dof']}_{ratio_col}.png"
        )
        plt.savefig(plot_dir / filename, dpi=200)
        plt.close()

    print(f"Saved ratio plots in: {plot_dir}")

make_ratio_plots(summary_with_ratios, "std_ratio_to_case_1", "STD ratio to damping case 1")
make_ratio_plots(summary_with_ratios, "mpm_ratio_to_case_1", "MPM ratio to damping case 1")

Saved ratio plots in: C:\Users\verav\Desktop\Studie\Afstuderen\PHASE2_ORCA\plots\damping_reduction
Saved ratio plots in: C:\Users\verav\Desktop\Studie\Afstuderen\PHASE2_ORCA\plots\damping_reduction


## 16. Snelle check: gemiddelde resultaten per damping case

Handig om snel te zien of de damping cases logisch effect hebben.

In [51]:
if not summary_with_ratios.empty:
    quick_check = (
        summary_with_ratios
        .groupby(["dof", "damping_case"], as_index=False)
        .agg(
            mean_std=("std", "mean"),
            mean_mpm_abs=("mpm_abs", "mean"),
            mean_std_ratio=("std_ratio_to_case_1", "mean"),
            mean_mpm_ratio=("mpm_ratio_to_case_1", "mean"),
        )
    )
    display(quick_check)
else:
    print("No summary results available.")

,dof,damping_case,mean_std,mean_mpm_abs,mean_std_ratio,mean_mpm_ratio
0,Heave,1,1.126754,2.859001,1.000000,1.000000
1,Heave,2,0.539632,1.369523,0.478927,0.479022
2,Heave,3,0.566464,1.438314,0.502740,0.503083
3,Heave,4,0.517409,1.313010,0.459203,0.459255
4,Pitch,1,10.934614,27.913851,1.000000,1.000000
5,Pitch,2,4.797830,12.291351,0.438775,0.440332
6,Pitch,3,4.758206,12.192906,0.435151,0.436805
7,Pitch,4,4.726510,12.110768,0.432252,0.433862
8,Roll,1,13.593214,34.568251,1.000000,1.000000
9,Roll,2,6.145077,15.723634,0.452069,0.454858
